[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rl-tools/rl-tools/blob/master/interface/python/hyperdrone/hyperdrone/examples/Overview.ipynb)

# HyperDrone Overview

The environment in five cells: three content-addressed ProcTHOR scenes, four drone models
flying in each, one simulation step, and every output modality the environment renders —
the shot of Figure 1 of the HyperDrone paper and its counterparts in two more scenes. The
paper's API appendix is generated from this notebook. `pip install hyperdrone` is the only prerequisite. The first construction of an
environment configuration JIT-compiles it; later constructions load it from the cache.

In [ ]:
import importlib.util
import site
import sysconfig
from pathlib import Path

checkout = next((path for path in [Path.cwd(), *Path.cwd().parents] if (path / "interface/python/hyperdrone/pyproject.toml").is_file()), None)
if checkout is not None:
    %pip install -q -e "{checkout}/interface/python/hyperdrone[examples]"
    site.addsitedir(sysconfig.get_paths()["purelib"])  # activates the editable install's .pth without a kernel restart
elif importlib.util.find_spec("hyperdrone") is None:  # fresh runtime (e.g. Colab)
    %pip install -q "hyperdrone[examples]"

The folded cell below holds only the false-color helpers of Figure 1 (depth, normals,
label palette, flow), the mosaic layout and a pose formatter.

In [ ]:
from textwrap import fill

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import hsv_to_rgb

import hyperdrone
from hyperdrone import render

hyperdrone.set_verbosity("warning")
%config InlineBackend.print_figure_kwargs = {"bbox_inches": "tight", "pad_inches": 0}
FLOW_MAX_PIXELS = 25.0


def pose(position, yaw_degrees):
    half = np.deg2rad(yaw_degrees) / 2
    return [*position, float(np.cos(half)), 0.0, 0.0, float(np.sin(half))]


def to_uint8(image):
    return image if image.dtype == np.uint8 else (np.clip(image, 0, 1) * 255).astype(np.uint8)


def depth_image(depth):
    finite = np.where(np.isfinite(depth), depth, np.nan)
    near = np.nanmin(finite, axis=(-2, -1), keepdims=True)  # per image
    far = np.nanmax(finite, axis=(-2, -1), keepdims=True)
    image = np.nan_to_num((far - finite) / (far - near) * 255).astype(np.uint8)
    return np.repeat(image[..., None], 3, axis=-1)


def normals_image(normals):
    valid = normals.any(axis=-1)
    image = np.zeros(normals.shape, dtype=np.uint8)
    image[valid] = np.rint((normals[valid] + 1) * 0.5 * 255)
    return image


def segmentation_image(labels, *, semantic=False):
    ids, inverse = np.unique(labels, return_inverse=True)
    palette = np.zeros((len(ids), 3), dtype=np.uint8)
    for index, label in enumerate(ids):
        if label == np.iinfo(np.uint32).max:
            continue  # a missed ray; instance ID zero is valid geometry
        if semantic and label == 0:
            palette[index] = 128  # undefined semantic class (the drone models carry none)
        else:
            hue = (int(label) * 0.618033988749895 + 0.08) % 1.0
            palette[index] = np.rint(hsv_to_rgb((hue, 0.65, 0.95)) * 255)
    return palette[inverse].reshape(*labels.shape, 3)


def flow_image(flow, status):
    hue = np.mod(np.arctan2(flow[..., 1], flow[..., 0]) / (2 * np.pi), 1)
    saturation = np.clip(np.linalg.norm(flow, axis=-1) / FLOW_MAX_PIXELS, 0, 1)
    return hsv_to_rgb(np.stack([hue, saturation, status == render.FlowStatus.VALID], axis=-1))


def mosaic(rows, columns, dpi=200):
    columns = {title: to_uint8(tiles) for title, tiles in columns.items()}
    size = next(iter(columns.values())).shape[1]
    gap, band = size // 128, size // 4  # tile gap; label band above and left of the tiles
    step, fontsize = size + gap, band / 2.5 * 72 / dpi  # two lines of text per band
    canvas = np.full((band + len(rows) * step - gap, band + len(columns) * step - gap, 3), 255, dtype=np.uint8)
    figure = plt.figure(figsize=(canvas.shape[1] / dpi, canvas.shape[0] / dpi), dpi=dpi)
    axis = figure.add_axes([0, 0, 1, 1])
    for column, (title, tiles) in enumerate(columns.items()):
        x = band + column * step
        axis.text(x + size / 2, band / 2, fill(title, 13), ha="center", va="center", fontsize=fontsize)
        for row, tile in enumerate(tiles):
            y = band + row * step
            canvas[y:y + size, x:x + size] = tile
    for row, label in enumerate(rows):
        axis.text(band / 2, band + row * step + size / 2, fill(label, 13), rotation=90, ha="center", va="center", fontsize=fontsize)
    axis.imshow(canvas, interpolation="none")
    axis.axis("off")
    plt.show()

## Scene and drone table

Scenes are GLB files addressed by content hash: the `Train200` catalog pins the 200 ProcTHOR
training scenes, three of which are used here, and `conta` fetches each once into a shared
cache. The drone table lists
the models that fly together in one instance, each a rigged GLB paired with the dynamics
model of the same name from the registry.

In [ ]:
import numpy as np
from hyperdrone import render
from hyperdrone.env import EnvConfig, MultiEnvironment
from hyperdrone.examples.data import rigged_model_path

catalog = render.datasets.Train200()
scenes = [catalog[f"ProcTHOR-Train-{index}"] for index in (1, 99, 72)]
MODELS = ("crazyflie_brushless", "arpl", "x500", "savagebee_pusher")
drones = [{"asset": rigged_model_path(name),
           "parameters": {"dynamics": {"dynamics": {"model": name}}}}
          for name in MODELS]

## Environment

`EnvConfig` is the compile-time part of an environment and its JIT key: resolution,
outputs, motion-blur samples and the agent layout. Each scene gets its own environment,
holding one instance of four mutually visible agents; agent 0 carries the only camera, and
its mount puts that camera at the pose of Figure 1. The `instance_segmentation` and `semantic_segmentation` outputs
label each pixel with its instance ID and its class ID. Each model's dynamics declare its rotor speed,
so the propellers spin at their physical rate.

In [ ]:
config = EnvConfig(preset="x500_fpv", num_agents=4, instances_per_environment=1,
                   num_environments=len(scenes), width=1024, height=1024,
                   motion_blur_samples=16, dataset="procthor",
                   output=("rgb", "depth", "instance_segmentation",
                           "semantic_segmentation", "normals", "flow"))
mount = {"position": [-0.2102, 0.2639, 0.3870],  # Figure 1's camera in
         "orientation": [0.6276, 0.2865, 0.2584, -0.6763]}  # agent 0 body frame
drones[0]["parameters"]["cameras"] = [{"fov": 80, "mount": {"pose": mount}}]

## Initialization

Runtime parameters are a partial dict merged over the nominal values. Here the
initialization distribution is pinned to zero angular velocity with the propellers at full
speed and a random initial velocity of up to 6 m/s per axis, which is what the flow tiles
will show.

In [ ]:
initial = {"guidance": 0, "max_angular_velocity": 0, "max_linear_velocity": 6.0,
           "min_rpm": 1, "max_rpm": 1}
parameters = {"dynamics": {"mdp": {"init": initial}}}
env = MultiEnvironment([scene.reference for scene in scenes], config=config,
                       drones=drones, seed=0, parameters=parameters)

## Reset and step

`reset` samples the scene placement and the poses; recorded spawn points, one per instance,
replay specific ones: the four poses of Figure 1 in the first scene and, in the others, the
camera drone at a new viewpoint with the other three rearranged in view, each with its
scene at the origin. `observe`
renders lazily, and `step` integrates 10 ms of dynamics for all agents. A policy would map
the observation to actions; here every agent applies its hover throttle from the resolved
drone table.

In [ ]:
formations = [  # (position, yaw in degrees) per agent
    [((-1.602, -3.994, 1.145), 65), ((-0.459, -4.226, 1.120), 30),
     ((-0.365, -5.578, 1.073), -35), ((-1.806, -4.461, 0.982), -20)],
    [((-13.976, -12.003, 1.103), 30), ((-13.768, -13.273, 0.667), 150),
     ((-14.423, -12.652, 0.718), -100), ((-13.466, -12.420, 0.642), 20)],
    [((-2.082, -13.649, 0.943), -139), ((-2.014, -11.854, 0.634), 45),
     ((-2.374, -12.688, 0.605), -150), ((-3.100, -13.230, 0.617), 80)]]
shots = [{"scene": scene.reference, "drone": [0, 1, 2, 3],
          "poses": [pose(position, yaw) for position, yaw in formation],
          "scene_translation": [0, 0, 0], "scene_yaw": [1, 0],
          "shutter_fraction": 0.2}
         for scene, formation in zip(scenes, formations)]
table = [d["parameters"]["dynamics"]["dynamics"] for d in env.drones]
hover = np.repeat([2 * t["hovering_throttle_relative"] - 1 for t in table], 4)
actions = np.tile(hover, (len(scenes), 1))  # what a policy would output
env.reset(spawn_points=shots)
env.observe()                               # first capture: the flow reference
env.step(actions)

## Outputs

Accessors return `(instances, views, H, W[, C])` snapshots and accept `out=` buffers
(CUDA arrays on an OptiX environment); `frames()` is the camera image behind the
observation. Each instance, here each scene, is one row of the mosaic. Flow is the displacement to the previous capture, valid where
`flow_status()` says so; the false-color mappings are those of Figure 1, with flow
saturating at 25 px.

In [ ]:
mosaic([scene.id for scene in scenes], {
    "RGB": env.frames(),
    "Depth": depth_image(env.depth()[:, 0]),
    "Surface normals": normals_image(env.normals()[:, 0]),
    "Semantic segmentation": segmentation_image(
        env.semantic_segmentation()[:, 0], semantic=True),
    "Instance segmentation": segmentation_image(
        env.instance_segmentation()[:, 0]),
    "Optical flow": flow_image(env.flow()[:, 0], env.flow_status()[:, 0]),
})

Flying the environment with a pilot, custom tasks and the C++ extension ladder are in
[HyperDrone.ipynb](HyperDrone.ipynb) and [Orbiter.ipynb](Orbiter.ipynb); the renderer and
the dynamics as standalone peers are in [Renderer.ipynb](Renderer.ipynb); throughput is
[Benchmark.ipynb](Benchmark.ipynb).